<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/06%20-%20Quantificadores%20e%20Predicados%20em%20Redes%20de%20Sensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 - Quantificadores e Predicados em Redes de Sensores
Validação de integridade e alarme da instrumentação via Lógica de Primeira Ordem (∀ e ∃).

In [ ]:
from typing import Dict, List, Callable

# 1. Universo de Sensores R-101
sensores: List[Dict] = [
    {"tag": "PT-101", "ativo": True,  "corrente_mA": 12.0, "valor": 2.1,  "min_safe": 0.5, "max_safe": 5.0},
    {"tag": "TT-101", "ativo": True,  "corrente_mA": 16.5, "valor": 78.5, "min_safe": 10.0, "max_safe": 85.0},
    {"tag": "AT-101", "ativo": True,  "corrente_mA": 14.0, "valor": 6.8,  "min_safe": 6.0, "max_safe": 8.0},
    {"tag": "LT-101", "ativo": True,  "corrente_mA": 18.2, "valor": 88.0, "min_safe": 10.0, "max_safe": 90.0},
    {"tag": "FT-101", "ativo": True,  "corrente_mA": 3.8,  "valor": 0.0,  "min_safe": 2.0,  "max_safe": 40.0}
]

# 2. Definição dos Predicados Lógicos
def ativo(s: Dict) -> bool:
    return s["ativo"]

def falha(s: Dict) -> bool:
    return s["corrente_mA"] < 4.0 or s["corrente_mA"] > 20.0

def critico(s: Dict) -> bool:
    if falha(s):
        return False
    return s["valor"] < s["min_safe"] or s["valor"] > s["max_safe"]

# 3. Implementação dos Quantificadores
def forall(universo: List[Dict], predicado: Callable[[Dict], bool]) -> bool:
    return all(predicado(x) for x in universo)

def exists(universo: List[Dict], predicado: Callable[[Dict], bool]) -> bool:
    return any(predicado(x) for x in universo)

# 4. Avaliação Lógica da Planta
rede_integra = forall(sensores, lambda s: ativo(s) and not falha(s))
existe_falha = exists(sensores, falha)
existe_alarme = exists(sensores, lambda s: ativo(s) and critico(s))
trip_sis = existe_falha or existe_alarme

# 5. Exibição dos Resultados
print("--- AVALIAÇÃO DA REDE DE SENSORES R-101 ---")
print(f"∀x [Ativo(x) ∧ ¬Falha(x)] -> Rede Íntegra: {rede_integra}")
print(f"∃x [Falha(x)]            -> Sensor em Falha: {existe_falha}")
print(f"∃x [Critico(x)]          -> Variável Fora de Faixa: {existe_alarme}")
print(f"TRIP DE SEGURANÇA (SIS)  -> Intertravamento: {trip_sis}")

sensores_falha = [s["tag"] for s in sensores if falha(s)]
print(f"Sensores com falha detectada: {sensores_falha}")
